# LEVEL 2 Overrepresentation EDA

This notebook audits the synthetic smishing dataset with a narrow focus on `LEVEL 2 – Leet nặng + tên riêng`, which currently dominates `synthetic_label_1.csv`.

The goal is not only to count the imbalance, but to determine whether `LEVEL 2` is:

- a realistic representation of Vietnamese smishing style,
- an artifact of prompt templates or generation batches,
- a taxonomy problem where several distinct transformations were collapsed into one level,
- or a data-quality issue that should be corrected before the final merge.

Primary input: `../synthetic/synthetic_label_1.csv`.

Optional output from this notebook: `level2_review_samples.csv`, a compact manual-review sample.

## Analysis Plan

1. Load and validate the current synthetic smishing data.
2. Quantify the global and per-category `level` distribution.
3. Compare observed vs expected counts to identify which categories drive the `LEVEL 2` surplus.
4. Measure surface obfuscation: digit ratio, symbol ratio, uppercase ratio, dot/dash insertion, leet-like substitutions, and random suffix codes.
5. Compare `LEVEL 2` against all other levels to see whether the label reflects distinct text behavior.
6. Detect likely template reuse using normalized skeletons.
7. Pull qualitative review samples, including high-confidence and suspicious `LEVEL 2` rows.
8. Convert findings into dataset decisions: rebalance, relabel, regenerate, or revise taxonomy/prompts.

In [ ]:
from pathlib import Path
import math
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_colwidth', 180)
pd.set_option('display.max_rows', 100)

DATA_PATH = Path('../synthetic/synthetic_label_1.csv')
REPORT_DIR = Path('.')
SAMPLE_OUT = REPORT_DIR / 'level2_review_samples.csv'

LEVEL2 = 'LEVEL 2 – Leet nặng + tên riêng'

df = pd.read_csv(DATA_PATH)
df['content'] = df['content'].fillna('').astype(str)

print(df.shape)
display(df.head())
display(df.dtypes)

## Basic Integrity Checks

These checks confirm that the notebook is analyzing the cleaned synthetic smishing file described in the handoff.

In [ ]:
required_cols = {'content', 'label', 'has_url', 'has_phone_number', 'sender_type', 'category', 'level'}
missing = required_cols - set(df.columns)
assert not missing, f'Missing columns: {missing}'

integrity = pd.DataFrame({
    'missing_values': df[list(required_cols)].isna().sum(),
    'unique_values': df[list(required_cols)].nunique(dropna=False),
})
display(integrity)

print('Label distribution:')
display(df['label'].value_counts(dropna=False).rename_axis('label').to_frame('count'))

print('Duplicate content rows:', df.duplicated('content').sum())

## Level Distribution

This section establishes the scale of overrepresentation. A balanced six-level design would put each level near 16.7%, but the current distribution should be evaluated against the intended generation design if a different target was planned.

In [ ]:
level_counts = df['level'].value_counts().rename_axis('level').to_frame('count')
level_counts['percent'] = 100 * level_counts['count'] / len(df)
display(level_counts)

ax = level_counts.sort_values('count').plot.barh(y='count', legend=False, figsize=(9, 4), color='#4C78A8')
ax.set_title('Synthetic smishing count by obfuscation level')
ax.set_xlabel('Rows')
ax.set_ylabel('')
for container in ax.containers:
    ax.bar_label(container, fmt='%d', padding=3)
plt.tight_layout()

## Category Contribution

A global imbalance can come from one category or from a broad generation policy. This cross-tab shows where `LEVEL 2` is concentrated.

In [ ]:
cat_level = pd.crosstab(df['category'], df['level'])
cat_level_pct = pd.crosstab(df['category'], df['level'], normalize='index') * 100

display(cat_level)
display(cat_level_pct.round(1))

level2_by_cat = cat_level.get(LEVEL2, pd.Series(dtype=int)).sort_values(ascending=False).to_frame('level2_count')
level2_by_cat['category_total'] = df['category'].value_counts()
level2_by_cat['level2_percent_within_category'] = 100 * level2_by_cat['level2_count'] / level2_by_cat['category_total']
level2_by_cat['share_of_all_level2'] = 100 * level2_by_cat['level2_count'] / level2_by_cat['level2_count'].sum()
display(level2_by_cat.round(1))

ax = level2_by_cat.sort_values('level2_count').plot.barh(y='level2_count', legend=False, figsize=(9, 4), color='#F58518')
ax.set_title('LEVEL 2 contribution by category')
ax.set_xlabel('LEVEL 2 rows')
ax.set_ylabel('')
plt.tight_layout()

## Observed vs Expected Level Counts

This residual table compares actual category-level counts against an independence baseline. Large positive residuals mean a category has more rows at that level than expected from the category size and level size alone.

Use this to identify where `LEVEL 2` is structurally over-assigned.

In [ ]:
observed = pd.crosstab(df['category'], df['level'])
expected = np.outer(observed.sum(axis=1), observed.sum(axis=0)) / observed.values.sum()
expected = pd.DataFrame(expected, index=observed.index, columns=observed.columns)
std_resid = (observed - expected) / np.sqrt(expected)

display(std_resid.round(2).sort_values(LEVEL2, ascending=False))

resid_long = std_resid.stack().rename('std_residual').reset_index()
display(resid_long.sort_values('std_residual', ascending=False).head(15))
display(resid_long.sort_values('std_residual', ascending=True).head(15))

## Text Feature Engineering

The metrics below are proxies, not ground truth. They are useful for finding rows whose assigned level does not match their surface form.

- `digit_ratio`: how much the text depends on numeric substitutions.
- `symbol_ratio`: punctuation and special-character density.
- `leet_char_count`: count of common leet substitution characters.
- `dot_dash_inside_token_count`: likely dot/dash insertion inside words.
- `random_suffix_flag`: generated-looking trailing code, common in synthetic examples.
- `vietnamese_diacritic_count`: whether heavy leet removed Vietnamese orthographic signal.
- `name_hint_count`: weak proxy for names or direct personal addressing.

In [ ]:
leet_chars = set('40135789@$!|+zZ')
vn_diacritic_re = re.compile(r'[àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđÀÁẠẢÃÂẦẤẬẨẪĂẰẮẶẲẴÈÉẸẺẼÊỀẾỆỂỄÌÍỊỈĨÒÓỌỎÕÔỒỐỘỔỖƠỜỚỢỞỠÙÚỤỦŨƯỪỨỰỬỮỲÝỴỶỸĐ]')
dot_dash_inside_token_re = re.compile(r'(?iu)\b\w+[.-]\w+(?:[.-]\w+)*\b')
url_re = re.compile(r'(?iu)\b(?:https?://|www\.|[a-z0-9-]+\.(?:com|vn|net|org|info|icu|top|xyz|site|me)\b)')
phone_like_re = re.compile(r'(?<!\d)(?:\+?84|0)\d{8,10}(?!\d)|(?<!\d)\d{3,5}(?!\d)')
random_suffix_re = re.compile(r'(?u)(?:\s|^)[A-Za-z]{1,3}\d[A-Za-z0-9]{1,4}\s*$')
name_hint_re = re.compile(r'(?iu)\b(?:anh|chị|chi|ông|ong|bà|ba|cô|co|chú|chu|quý khách|quy khach|khách hàng|khach hang|bạn|ban)\b')

def char_stats(text: str) -> pd.Series:
    chars = [c for c in text if not c.isspace()]
    n = max(len(chars), 1)
    letters = [c for c in chars if c.isalpha()]
    return pd.Series({
        'char_count': len(text),
        'token_count': len(re.findall(r'\S+', text)),
        'digit_count': sum(c.isdigit() for c in chars),
        'digit_ratio': sum(c.isdigit() for c in chars) / n,
        'symbol_count': sum((not c.isalnum()) for c in chars),
        'symbol_ratio': sum((not c.isalnum()) for c in chars) / n,
        'upper_ratio': (sum(c.isupper() for c in letters) / max(len(letters), 1)),
        'leet_char_count': sum(c in leet_chars for c in chars),
        'leet_char_ratio': sum(c in leet_chars for c in chars) / n,
        'vietnamese_diacritic_count': len(vn_diacritic_re.findall(text)),
        'dot_dash_inside_token_count': len(dot_dash_inside_token_re.findall(text)),
        'url_like_count': len(url_re.findall(text)),
        'phone_like_count': len(phone_like_re.findall(text)),
        'random_suffix_flag': int(bool(random_suffix_re.search(text))),
        'name_hint_count': len(name_hint_re.findall(text)),
    })

features = df['content'].apply(char_stats)
eda = pd.concat([df, features], axis=1)
display(eda.head())

## Feature Summary by Level

This view checks whether `LEVEL 2` is meaningfully distinct. If it looks similar to `LEVEL 1`, `LEVEL 3`, or `LEVEL 4`, the taxonomy may be too ambiguous or the generation process may have blended transformations.

In [ ]:
feature_cols = [
    'char_count', 'token_count', 'digit_ratio', 'symbol_ratio', 'upper_ratio',
    'leet_char_ratio', 'vietnamese_diacritic_count', 'dot_dash_inside_token_count',
    'url_like_count', 'phone_like_count', 'random_suffix_flag', 'name_hint_count'
]

summary_by_level = eda.groupby('level')[feature_cols].agg(['mean', 'median']).round(3)
display(summary_by_level)

plot_cols = ['digit_ratio', 'symbol_ratio', 'leet_char_ratio', 'dot_dash_inside_token_count', 'random_suffix_flag', 'name_hint_count']
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), plot_cols):
    eda.boxplot(column=col, by='level', ax=ax, rot=80)
    ax.set_title(col)
    ax.set_xlabel('')
fig.suptitle('Surface feature distributions by level')
plt.tight_layout()

## LEVEL 2 vs Non-LEVEL 2

This binary comparison is useful for deciding whether the overrepresented class has a coherent signature. Weak separation suggests relabeling or taxonomy revision; strong separation suggests the issue is mainly distributional.

In [ ]:
eda['is_level2'] = eda['level'].eq(LEVEL2)

binary_summary = eda.groupby('is_level2')[feature_cols].agg(['mean', 'median', 'std']).round(3)
display(binary_summary)

effect_rows = []
for col in feature_cols:
    a = eda.loc[eda['is_level2'], col].astype(float)
    b = eda.loc[~eda['is_level2'], col].astype(float)
    pooled = math.sqrt(((a.var(ddof=1) + b.var(ddof=1)) / 2)) if len(a) > 1 and len(b) > 1 else np.nan
    effect_rows.append({
        'feature': col,
        'level2_mean': a.mean(),
        'non_level2_mean': b.mean(),
        'mean_diff': a.mean() - b.mean(),
        'cohens_d_approx': (a.mean() - b.mean()) / pooled if pooled and not np.isnan(pooled) else np.nan,
    })

effect_df = pd.DataFrame(effect_rows).sort_values('cohens_d_approx', key=lambda s: s.abs(), ascending=False)
display(effect_df.round(3))

## Category-Specific LEVEL 2 Behavior

`LEVEL 2` may not mean the same thing in every category. For example, gambling and crypto rows may use more URL/domain patterns, while benefit-scam rows may use more official-language templates.

In [ ]:
level2 = eda[eda['is_level2']].copy()

level2_cat_features = level2.groupby('category')[feature_cols].mean().round(3)
level2_cat_features['rows'] = level2['category'].value_counts()
display(level2_cat_features.sort_values('rows', ascending=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['digit_ratio', 'symbol_ratio', 'leet_char_ratio']):
    level2.boxplot(column=col, by='category', ax=ax, rot=80)
    ax.set_title(f'LEVEL 2 {col} by category')
    ax.set_xlabel('')
fig.suptitle('')
plt.tight_layout()

## Template Reuse and Generation Artifacts

Heavy overrepresentation is more concerning if the level is also template-heavy. The skeleton below removes common volatile parts such as URLs, phone-like numbers, and trailing random codes. High skeleton duplication means the dataset may teach models a small set of synthetic templates rather than robust smishing cues.

In [ ]:
def strip_accents(text: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')

def skeletonize(text: str) -> str:
    x = strip_accents(text.lower())
    x = url_re.sub('<URL>', x)
    x = phone_like_re.sub('<NUM>', x)
    x = random_suffix_re.sub(' <SUFFIX>', x)
    x = re.sub(r'\d+', '<NUM>', x)
    x = re.sub(r'[^a-z<>]+', ' ', x)
    x = re.sub(r'\s+', ' ', x).strip()
    return x

eda['skeleton'] = eda['content'].map(skeletonize)
level2 = eda[eda['is_level2']].copy()

skeleton_stats = eda.groupby(['level', 'skeleton']).size().rename('count').reset_index()
top_skeletons = skeleton_stats.sort_values('count', ascending=False).head(30)
display(top_skeletons)

reuse_by_level = skeleton_stats.groupby('level')['count'].agg(
    rows='sum',
    unique_skeletons='count',
    max_skeleton_reuse='max',
    skeletons_reused_3plus=lambda s: int((s >= 3).sum()),
    rows_in_reused_3plus=lambda s: int(s[s >= 3].sum()),
)
reuse_by_level['rows_per_skeleton'] = reuse_by_level['rows'] / reuse_by_level['unique_skeletons']
reuse_by_level['pct_rows_in_reused_3plus'] = 100 * reuse_by_level['rows_in_reused_3plus'] / reuse_by_level['rows']
display(reuse_by_level.round(2).sort_values('pct_rows_in_reused_3plus', ascending=False))

## Suspicious LEVEL 2 Rows

These rows are useful for manual audit. They are still labeled `LEVEL 2`, but their surface metrics suggest they may be closer to another level or may reflect synthetic artifacts.

In [ ]:
level2_thresholds = {
    'low_leet': level2['leet_char_ratio'].quantile(0.10),
    'high_symbol': level2['symbol_ratio'].quantile(0.90),
    'high_dotdash': level2['dot_dash_inside_token_count'].quantile(0.90),
}
level2_thresholds

suspicious_level2 = level2[
    (level2['leet_char_ratio'] <= level2_thresholds['low_leet']) |
    (level2['dot_dash_inside_token_count'] >= max(1, level2_thresholds['high_dotdash'])) |
    (level2['random_suffix_flag'].eq(1))
].copy()

review_cols = [
    'content', 'category', 'level', 'has_url', 'has_phone_number', 'sender_type',
    'digit_ratio', 'symbol_ratio', 'leet_char_ratio', 'dot_dash_inside_token_count',
    'random_suffix_flag', 'name_hint_count', 'skeleton'
]

display(suspicious_level2[review_cols].sort_values(['category', 'leet_char_ratio']).head(80))

## Representative Samples for Manual Review

This sample intentionally mixes typical, low-leet, high-symbol, and high-template-reuse examples. Review these rows by hand before deciding whether to relabel or regenerate.

In [ ]:
rng = np.random.default_rng(42)

sample_parts = []
for category, g in level2.groupby('category'):
    sample_parts.append(g.sample(min(5, len(g)), random_state=42))
    sample_parts.append(g.nsmallest(min(5, len(g)), 'leet_char_ratio'))
    sample_parts.append(g.nlargest(min(5, len(g)), 'symbol_ratio'))

review_sample = pd.concat(sample_parts, ignore_index=False).drop_duplicates().copy()
review_sample = review_sample[review_cols].sort_values(['category', 'leet_char_ratio', 'symbol_ratio'])
review_sample.to_csv(SAMPLE_OUT, index=False, encoding='utf-8-sig')

print(f'Wrote {len(review_sample)} rows to {SAMPLE_OUT}')
display(review_sample.head(120))

## Decision Framework

Use the EDA outputs to decide one of the following actions before final merging:

### Keep `LEVEL 2` as-is only if

- `LEVEL 2` has a distinct feature profile from neighboring levels,
- overrepresentation is intentional and thesis-justified,
- template reuse is not excessive,
- qualitative review confirms that the leet style resembles real Vietnamese smishing rather than synthetic noise.

### Relabel part of `LEVEL 2` if

- low-leet rows look like `LEVEL 0` or `LEVEL 1`,
- dot/dash-heavy rows look like `LEVEL 3`,
- symbol-heavy rows look like `LEVEL 4`,
- suffix-code artifacts are not part of the intended taxonomy.

### Regenerate or downsample if

- `LEVEL 2` is mostly caused by prompt batching,
- a few categories dominate because generation prompts defaulted to heavy leet,
- repeated skeletons are high enough to reduce dataset diversity,
- the final training setup needs balanced obfuscation exposure.

### Revise taxonomy if

- `Leet nặng` and `tên riêng` are not consistently co-occurring,
- Vietnamese diacritic loss, numeric substitution, name insertion, and random suffixes behave like separate transformations,
- the current levels do not form a monotonic difficulty scale.

## Notes for Thesis Write-Up

The key distinction to document is whether level imbalance is a data-design choice or a generation artifact. If retained, justify it with observed scam-style evidence or a rationale tied to robustness testing. If corrected, document the correction rule, the number of affected rows, and why the corrected distribution better supports evaluation validity.